## Harvest - kWh

#### <span style="color:red; font-weight:bold"> Instructions for Using this Jupyter Notebook:</span>
This Notebook is using end point value minus start point value of kWh to calculate the kWh within one fiscal year, for both the annual total and each calendar month.
- also finds special meters and makes needed corrections to the data prior

<span style="color:royalblue">(0) Install Python packages (one-time setup):</span>
- In terminal, run: **pip3 install scikit-learn**

<span style="color:royalblue">(1) Project layout (already set up):</span>
- This notebook lives in **notebooks/**, and **modules/data_clean_TEST.py** holds the functions it imports as `dc`.

<span style="color:royalblue">(2) Place these input files in **data/extracts/** (already done)</span>
- **meter_info.csv**: meter → building and end_use (main/submeter), exported from the database
- the raw interval-reading file for this run (e.g. **harvest_kwh_15min_260504-260713.csv**), set its name as `var_file` in Section 1
- **special_meter_candidates.csv**: reviewed special-meter windows (auto-generated/updated by the notebook; edit `solution` to `div100`, `remove`, or `broken` for rows that need a correction)
- **running_list_broken_meters.csv**: known broken intervals carried over between runs
- **special_meters_corrections_master_sheet.csv**: master sheet of approved offical corrections (auto-generated/updated by the notebook)

<span style="color:royalblue">(3) Modify the **Parameters in Section 1** as needed before running this Notebook:</span> 
- Time range, FY, file names, etc.

<span style="color:royalblue">(4) Run the Notebook:</span> 
- Set **Checked = False** & **Clear_Candidates = True** and run it once.
- Check **all_meters_plots.pdf** and **review_special_meters_plots.pdf** in **data/outputs/plots/**.
- If there are any unusual periods in the meter reading data, record (and approve) the correction in **special_meter_candidates.csv** (or **running_list_broken_meters.csv**) in **data/extracts/**.
- If no problem, set **Checked = True** & **Clear_Candidates = False** and run the notebook again.

<span style="color:royalblue">**Use R² to define Special Meters (R² < 0.9):**</span> 

- **R² (Coefficient of Determination) Definition:** Measures how closely the meter’s kWh readings follow a perfect linear increase (straight) line over time using linear regression.

- **R² Values and Meaning:**
    - R² ≥ 0.9: Excellent — the meter closely follows the expected linear trend.

    - 0.5 ≤ R² < 0.9: Moderate — the meter shows noticeable deviations; may have some irregular readings.

    - 0 < R² < 0.5: Poor — the meter data is highly irregular.

    - R² = 0: All values missing — no valid data.

    - R² = -1: Missing points at start or end of the fiscal year; scaling applied if enough points exist (> 5 months).

    - R² = -2: Stuck points at start or end of the fiscal year; scaling applied.
 
    - R² = -3: Meter restarts at least 1 time.

#### Files used by this notebook

| File | Location | Purpose |
|---|---|---|
| `harvest_kwh_15min_XXXXXX-XXXXXX.csv` | data/extracts/ | Source meter readings for the run. Set the filename using `var_file` in Section 1. |
| `meter_info.csv` | data/extracts/ | Meter metadata, including building assignments and meter types. Used to select meters and organize results. |
| `special_meter_candidates.csv` | data/extracts/ | Generated review file containing candidate issue intervals and review decisions. Review these rows and record corrections using the `solution` and `approved` fields. |
| `running_list_broken_meters.csv` | data/extracts/ | Known broken-meter intervals used when assembling corrections and reviewing meter data. |
| `special_meters_corrections_ master_sheet.csv` | data/outputs/ | Corrections assembled for application during the run, including approved candidate corrections and broken-meter intervals. |
| `removed_special_meter_data.csv` | data/outputs/ | Readings captured from `remove` and `broken` correction windows before special-meter corrections are applied. This file is overwritten, not appended to. |


Annual and monthly result files are saved separately in `data/outputs/annual_monthly/`. Review plots are saved in `data/outputs/plots/`.

### 1. Parameters

In [ ]:
############ CHANGE PARAMETERS AS NEEDED #############

Checked = False  # Set to True after you review the special_meters plot.
Insert = False   # Leave False for now unless you want to push results into the annual kwh sheet.

# Set to True to delete previous candidate files before this run 
# (sync approved rows first, otherwise you will lose them)
Clear_Candidates = True

# Time Range: Select One Fiscal Year
start_time = "2025-07-01 00:00:00" #"2025-07-23 09:40:50" 
end_time = "2026-07-20 12:45:00" #"2026-07-13 10:45:00" 
# requested_start_time = "2025-07-23 09:40:50"
# requested_end_time = "2025-10-17 11:39:02"

FY = "_fy26"

######################################################

In [ ]:

# Data Directories
input_dir = "../data/extracts/"  # directory for raw data files & other input files

output_dir = "../data/outputs/"  # directory for data outputs (different from input_dir)

plot_dir = "../data/outputs/plots/"  # directory for plot outputs

annual_monthly_dir = "../data/outputs/annual_monthly/"  # directory for annual and monthly check outputs


# Variable
var = 'kwh'

# current longest == harvest_kwh_15min_250723-260713
var_file = input_dir + 'harvest_kwh_15min_250701-260713.csv' #'fy26_aurora_data.csv' #'harvest_kwh_15min_250723-260713.csv' # data file 0, should contain only interval meter readings
meter_info_file = input_dir + "meter_info.csv"  # contains all meter information

# Review/source files
meter_issues_candidates_file = input_dir + "special_meter_candidates.csv"  # auto-generated review file
broken_meters_file = input_dir + "running_list_broken_meters.csv"  # auto-added broken intervals source

# Output Special Meter Files
meter_corrections_file = output_dir + "special_meters_corrections_master_sheet.csv"  # official + auto-broken used for this run
removed_special_meter_data_file = output_dir + "removed_special_meter_data.csv"  # raw data captured inside remove/broken windows

# Output Figures
all_meters_plot = plot_dir + "all_meters_plots" + FY + ".pdf"
review_overlay_plot = plot_dir + "review_special_meters_plots" + FY + ".pdf"

######################################################
if Insert:
    insert_sheet = input_dir + FY + "_annual_" + var + ".csv"
######################################################

# Annual Output Files
meter_annual_csv = annual_monthly_dir + "meter_annual_" + var + FY + ".csv"  # annual kwh usage for each meter
building_annual_csv = annual_monthly_dir + "building_annual_" + var + FY + ".csv"  # annual kwh usage for each building
scaling_annual_detail_csv = annual_monthly_dir + "meter_annual_scaling_detail" + FY + ".csv"

# Monthly Output Files
meter_monthly_csv = annual_monthly_dir + "meter_monthly_" + var + FY + ".csv"
monthly_pct_scaled_csv = annual_monthly_dir + "meter_monthly_pct_scaled" + FY + ".csv"
monthly_long_csv = annual_monthly_dir + "meter_monthly_long" + FY + ".csv"  # tidy format for Tableau: meter_name, month, kwh, pct_estimated
scaling_monthly_detail_csv = annual_monthly_dir + "meter_monthly_scaling_detail" + FY + ".csv"


annual_monthly_check_csv = annual_monthly_dir + "meter_check_annual_vs_monthly" + FY + ".csv"

# Minimum valid coverage required before a missing-edge month is scaled.
# 0.50 means at least half of that monthly period must contain usable data.
monthly_min_valid_fraction = 0.50


# Exclude: Meters of buildings equipped with PV and Student Health
meters_with_pv = [
    'bachman_hall_main',
    'campus_ctr_main',
    'dance_bldg_main',
    'gartley_hall_main',
    'warrior_rec_ctr_main'
]
meters_excluded = meters_with_pv + ['student_health_main']  # student_health data is in vitality_v5


# Valid Data Min Length
valid_len = 5*30*96  # A meter should have at least 5-months valid data within 1 fiscal year

# Parameters for data cleaning - no need to change for now
r2_threshold = 0.9

# If a meter restarts more than 5 times in a fiscal year, treat it as a Special Meter
restarts_thres = 5

# Data Frequency
freq = '15min'

# Schema
schema = 'harvest'
#schema = 'aurora_v4'


### 2. Imports

In [42]:
%load_ext autoreload
%autoreload 2

import os, sys
import pandas as pd
import numpy as np

import importlib
sys.path.append(os.path.abspath('..'))
import modules.data_clean_TEST as dc # import self-defined module
importlib.reload(dc)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


<module 'modules.data_clean_TEST' from '/Users/cassiehuber/Documents/GitHub/harvest_kwh_prep/modules/data_clean_TEST.py'>

##### <span style="color:royalblue">Clear Previous Candidates (optional):</span>
- If Clear_Candidates = True, deletes `special_meter_candidates.csv` and `special_meter_candidates_restarts.csv` before the rest of the notebook runs, so this run's candidates are generated from scratch

In [43]:
if Clear_Candidates:
    dc.clear_candidate_files(meter_issues_candidates_file)

Cleared previous candidate file: ../data/extracts/special_meter_candidates.csv
Cleared previous candidate file: ../data/extracts/special_meter_candidates_restarts.csv


### 3. Load Data

In [ ]:
# Read variable data file
raw_df = pd.read_csv(var_file, low_memory=False)

raw_df["datetime"] = pd.to_datetime(raw_df["datetime"])
raw_df.head()

,datetime,meter_name,meter_reading
0,2025-07-01 00:00:00,admin_serv_1,1369563.0
1,2025-07-01 00:15:00,admin_serv_1,1369566.0
2,2025-07-01 00:30:00,admin_serv_1,1369570.0
3,2025-07-01 00:45:00,admin_serv_1,1369573.0
4,2025-07-01 01:00:00,admin_serv_1,1369576.0


### 4. Reshape & Filter

In [45]:
# Sort by meter_name and datetime
raw_df = raw_df.sort_values(by=['meter_name', 'datetime']).reset_index(drop=True)

# Pivot table with every meter as one column
pivoted_df = raw_df.pivot(index='datetime', columns='meter_name', values='meter_reading').reset_index()

# Fill missing timestamps
full_df = dc.fill_missing_timestamps(pivoted_df, freq)

full_df.head()


,datetime,admin_serv_1,admin_serv_2_main,ag_engineering_main,ag_engineering_mcc,ag_science_main_1,ag_science_main_2,ag_science_mcc,andrews_amp_main,archtecture_main,...,sherman_main_2,spalding_hall_main,st_john_plant_science_main,stan_sheriff_ctr_main_1,stan_sheriff_ctr_main_2,student_health_main,transportation_srvc_main,univ_high_school_3_main,webster_hall_main,wist_annex_1_main
0,2025-07-01 00:00:00,1369563.0,389451.0,0.0,0.0,1084104.0,8674662.0,12501991.0,5499.0,306071.0,...,3816518.0,10146189.0,31495268.0,30240901.0,17489080.0,117776.0,188461.0,680094.0,4463458.0,629149.0
1,2025-07-01 00:15:00,1369566.0,389451.0,0.0,0.0,1084127.0,8674709.0,12502018.0,5499.0,306078.0,...,3816524.0,10146218.0,31495328.0,30240918.0,17489087.0,117778.0,188461.0,680096.0,4463464.0,629152.0
2,2025-07-01 00:30:00,1369570.0,389452.0,NaN,NaN,1084149.0,8674756.0,12502044.0,NaN,306085.0,...,3816531.0,10146248.0,31495389.0,30240934.0,17489093.0,117779.0,188462.0,680098.0,4463471.0,629155.0
3,2025-07-01 00:45:00,1369573.0,NaN,NaN,NaN,1084172.0,8674804.0,12502070.0,NaN,306093.0,...,3816538.0,10146277.0,31495448.0,30240950.0,17489099.0,117780.0,NaN,680100.0,4463477.0,629158.0
4,2025-07-01 01:00:00,1369576.0,389453.0,NaN,NaN,1084194.0,8674850.0,12502097.0,NaN,306100.0,...,3816545.0,10146306.0,31495506.0,30240967.0,17489104.0,117782.0,188463.0,680102.0,4463483.0,629161.0


##### <span style="color:royalblue">Filter Meters and Time Range:</span>

In [46]:
### Filter One: Retain only main and sub meters and filter out others and PV meters ###

filtered_df, meter_type_by_meter, selected_meter_info = (
    dc.select_meter_types_from_info(
        full_df,
        meter_info_file,
        allowed_end_uses=("main", "submeter"),
        excluded_meters=meters_excluded,
    )
)

# The same map is passed to the plot functions so titles show
# meter_name [main] or meter_name [submeter]
selected_meter_info.head()

Selected 91 meters: 74 main, 17 submeter.


,meter_name,end_use
0,parking_struct_ph_i_main,main
1,bachman_hall_annex,main
2,edmondson_hall_main,main
3,gilmore_hall_main_a,main
4,gilmore_hall_main_b,main


In [47]:
### Filter Two: Retain only the selected analysis-window data ###

# resolve the requested times to timestamps that actually exist
# in the filled index before slicing and before endpoint calculations.
start_time, end_time = dc.resolve_analysis_window(
    filtered_df.index,
    start_time,
    end_time,
)

print("Resolved start_time:", start_time)
print("Resolved end_time:", end_time)

data = filtered_df.loc[start_time:end_time, :].copy()
data.index = pd.to_datetime(data.index)

# Initial Data Cleaning: Replace all 0s with NaN in the entire DataFrame 
data = data.replace(0, np.nan)

data.head(2)


Resolved start_time: 2025-07-01 00:00:00
Resolved end_time: 2026-07-13 10:45:00


,admin_serv_1,admin_serv_2_main,ag_engineering_main,ag_engineering_mcc,ag_science_main_1,ag_science_main_2,ag_science_mcc,andrews_amp_main,archtecture_main,bachman_hall_annex,...,sherman_main_1,sherman_main_2,spalding_hall_main,st_john_plant_science_main,stan_sheriff_ctr_main_1,stan_sheriff_ctr_main_2,transportation_srvc_main,univ_high_school_3_main,webster_hall_main,wist_annex_1_main
datetime,,,,,,,,,,,,,,,,,,,,,
2025-07-01 00:00:00,1369563.0,389451.0,NaN,NaN,1084104.0,8674662.0,12501991.0,5499.0,306071.0,40521.0,...,3548717.0,3816518.0,10146189.0,31495268.0,30240901.0,17489080.0,188461.0,680094.0,4463458.0,629149.0
2025-07-01 00:15:00,1369566.0,389451.0,NaN,NaN,1084127.0,8674709.0,12502018.0,5499.0,306078.0,40522.0,...,3548723.0,3816524.0,10146218.0,31495328.0,30240918.0,17489087.0,188461.0,680096.0,4463464.0,629152.0


### 5. Special Meter Review loop:

##### <span style="color:royalblue">Sync Meter Corrections Master Sheet:</span>
Update or build (if not already exists) the master correction workbook of harvest kwh meter readings for this run

NOTE:
- The master corrections sheet is a broader historical/reference log.
- It may contain meters not present in the current raw data for this run.
- Only corrections for meters in the current dataset are actually applied.

In [48]:
# Master sheet combines:
# - reviewed rows already in special_meters_candidates.xlsx
# - approved candidate rows (approved == 1)
# - broken-meter rows from the running broken workbook
master_df = dc.sync_meter_corrections_master_sheet(
    meter_issues_candidates_file,
    broken_meters_file,
    meter_corrections_file,
    start_time,
    end_time,
)

Master correction file saved to ../data/outputs/special_meters_corrections_master_sheet.csv


##### <span style="color:royalblue">Export Removed/Broken Raw Data:</span>
- Save the raw readings that fall inside `remove` / `broken` correction windows before applying corrections.
- Note: overwrites file, not appending

In [49]:
# Export raw data inside remove/broken correction windows before corrections are applied
removed_special_meter_data = dc.export_removed_special_meter_data(
    data,
    meter_corrections_file,
    removed_special_meter_data_file,
)
removed_special_meter_data.head(2)


Removed special meter data exported to ../data/outputs/removed_special_meter_data.csv


,datetime,meter_name,meter_reading,solution,correction_start,correction_end,issue_type/status,description
0,2026-05-05 11:00:00,ag_engineering_mcc,667453.0,remove,2026-03-20,NaT,broken,
1,2026-05-05 11:15:00,ag_engineering_mcc,667454.0,remove,2026-03-20,NaT,broken,


##### <span style="color:royalblue">Correct Special Meters:</span>
- If `special_meters_corrections_master_sheet.csv` exists, its reviewed corrections are applied first.
- Also auto-generates `meter_issues_candidates.csv` after special meters are detected.

In [50]:
# Apply corrections from the master sheet
data_corrected = dc.apply_special_meter_corrections(data, meter_corrections_file)
data_corrected.head(2)


,admin_serv_1,admin_serv_2_main,ag_engineering_main,ag_engineering_mcc,ag_science_main_1,ag_science_main_2,ag_science_mcc,andrews_amp_main,archtecture_main,bachman_hall_annex,...,sherman_main_1,sherman_main_2,spalding_hall_main,st_john_plant_science_main,stan_sheriff_ctr_main_1,stan_sheriff_ctr_main_2,transportation_srvc_main,univ_high_school_3_main,webster_hall_main,wist_annex_1_main
datetime,,,,,,,,,,,,,,,,,,,,,
2025-07-01 00:00:00,1369563.0,389451.0,NaN,NaN,1084104.0,8674662.0,12501991.0,5499.0,306071.0,40521.0,...,3548717.0,3816518.0,10146189.0,31495268.0,30240901.0,17489080.0,188461.0,680094.0,4463458.0,629149.0
2025-07-01 00:15:00,1369566.0,389451.0,NaN,NaN,1084127.0,8674709.0,12502018.0,5499.0,306078.0,40522.0,...,3548723.0,3816524.0,10146218.0,31495328.0,30240918.0,17489087.0,188461.0,680096.0,4463464.0,629152.0


##### <span style="color:royalblue">Find Special Meters:</span>

In [51]:
# Find special meters (R² < 0.9) after applying any manualy approved corrections
df_special_meters, df_restarts = dc.find_special_meters(data_corrected, r2_threshold)

##### <span style="color:royalblue">Autocreate special meter candidates:</span>
Update the candidate meter issues workbook for review for this run
- keeps unresolved rows already in the candidate file
- removes rows where approved == 1
- writes newly detected candidate rows from the corrected data (including "special meters")
- excludes active broken meter rows from the candidate workbook

Output:
- prints a summary of detected special meters
- if restart rows exist, creates `special_meter_candidates_restarts.csv`
- creates `special_meter_candidates.csv` with interval candidate rows

In [52]:
# update special meter candidates file for review
# - includes unresolved rows, detected candidate timeframes, and summary derived review rows
df_meter_issues_candidates = dc.update_special_meter_candidates_workbook(
    data_corrected,
    meter_issues_candidates_file,
    broken_meters_file,
    start_time,
    end_time,
    df_bad_meters=df_special_meters,
    df_restarts=df_restarts,
)

df_meter_issues_candidates.head(2)


Restart file saved to ../data/extracts/special_meter_candidates_restarts.csv
Candidate review file saved to ../data/extracts/special_meter_candidates.csv

Special meter summary:
                 meter_name      r2                                          info
               admin_serv_1      -3                               restart 1 times
        ag_engineering_main      -3                               restart 1 times
         ag_engineering_mcc      -3                               restart 1 times
         bachman_hall_annex      -3                               restart 1 times
          building_037_main      -3                               restart 1 times
   building_1171a_to_f_main      -3                               restart 1 times
         building_1171f_cds      -3                               restart 1 times
        env_protection_main      -3                               restart 2 times
        gilmore_hall_main_b      -3                               restart 1 times
  

,meter_name,solution,start_datetime,end_datetime,issue_type,description,suggestion,r2,approved
0,admin_serv_1,,2025-11-02 01:15:00,2025-11-02 01:30:00,restart_or_drop,Negative jump of -1.916399999987334.,review_restart,-3,0
1,ag_engineering_main,,2025-11-02 01:30:00,2025-11-02 01:45:00,restart_or_drop,Negative jump of -1.9677999999839813.,review_restart,-3,0


In [53]:
print("special meters dataframe info:")
print("data_corrected shape:", data_corrected.shape)
print("index type:", type(data_corrected.index))
print("index min:", data_corrected.index.min())
print("index max:", data_corrected.index.max())

special meters dataframe info:
data_corrected shape: (36236, 91)
index type: <class 'pandas.DatetimeIndex'>
index min: 2025-07-01 00:00:00
index max: 2026-07-13 10:45:00


##### <span style="color:royalblue">Plot All Special Meters:</span>
Plot review meters with overlay windows
- Red spans  = broken meter intervals from the broken meter workbook `running_list_broken_meters.csv`
- Blue spans = candidate intervals from the candidate workbook `special_meter_candidates.csv`
- Black dashed lines = restart or drop event from the candidate workbook
- More blue spans = overlapping candidate intervals
- Purple spans = both red and blue spans

The plot annotation box shows:
- issue_type first
- then R² text if present

In [54]:
dc.plot_review_meters_with_overlays(
    data,
    meter_issues_candidates_file,
    broken_meters_file,
    review_overlay_plot,
    ylabel=var,
    meter_type_map=meter_type_by_meter,
)   


Review overlay plots saved to ../data/outputs/plots/review_special_meters_plots_fy26.pdf


42

##### <span style="color:royalblue">Plot All Meters:</span>
- PDF of all meters (special and non special) after corrections applied by master sheet

In [55]:
dc.plot_all_meters_to_pdf(
    data_corrected,
    all_meters_plot,
    ylabel=var,
    meter_type_map=meter_type_by_meter,
)

All-meter plots saved to ../data/outputs/plots/all_meters_plots_fy26.pdf


91

#### <span style="color:red">Check</span>
If have not already, inspect the `special_meter_candidates.csv` and pdfs (`review_special_meters_plots.pdf`) then set Checked == True.

In [56]:
if not Checked:
    raise SystemExit()

SystemExit: 

/Users/cassiehuber/miniconda3/envs/harvest/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


#### <span style="color:red">!!! Note: Check spcieal_meter_candidates.csv & review_special_meters_plots_fyXX.pdf first before run the following cells !!!</span>

The annual and monthly scaling-detail files include:

| Field | What to review |
|---|---|
| `raw_difference` | The reference difference returned by the calculation function and used to calculate `estimated_kwh`. Do not assume this was calculated from untouched source readings, because the function receives `data_corrected`. |
| `difference` | The final calculated kWh value used in the annual or monthly results. |
| `estimated_kwh` | The difference between `difference` and `raw_difference`, calculated explicitly by the notebook. |
| `R²` and `info` | The score or status and accompanying information returned by the calculation function. |
| `% scaled` | The scaling information returned by the calculation function. Review it alongside `info` and `estimated_kwh`, rather than interpreting it alone. |


### 6. Calculate (End - Start) Difference for Annual kWh

Output:
- `meter_annual_scaling_detail_fyXX.csv`
- `meter_annual_kwh_fyXX.csv`
- `annual_monthly/building_annual_kwh_fyXX.csv`


In [ ]:
### Compute kWh difference between End point and Start point; Scaling applied for some special meters ###

# Step 1: Compute differences
result_df = dc.compute_meter_differences(
    data_corrected,
    start_time,
    end_time,
    df_special_meters,
    df_restarts,
    valid_len=valid_len,
    r2_threshold=r2_threshold,
    restarts_thres=restarts_thres,
)

################ SCALING DETAIL ################
scaling_detail = result_df.reset_index()[[
    "meter_name", "raw_difference", "difference", "R²", "info", "% scaled"
]]

scaling_detail["estimated_kwh"] = (
    scaling_detail["difference"] - scaling_detail["raw_difference"]
)

scaling_detail.to_csv(scaling_annual_detail_csv, index=False)
print("Total estimated kWh:", scaling_detail["estimated_kwh"].sum())
print(f"Annual scaling detail saved to {scaling_annual_detail_csv}")
################################################

# Export meter-level differences only
df_all = result_df.reset_index()[["meter_name", "difference"]].copy()
df_all.rename(columns={"difference": f"annual_{var}"}, inplace=True)
df_all[f"annual_{var}"] = df_all[f"annual_{var}"].round(1)
df_all.to_csv(meter_annual_csv, index=False)

# Step 2: Export all meters' differences -> annual kWh usage (CSV, rounded 1 decimal)
df_all = dc.export_meter_differences(result_df, meter_info_file, meter_annual_csv, var=var)

# Step 3: Export building-level differences -> annual kWh per building (CSV)
# sum calculated with main meters only
df_building_sum = dc.export_building_differences(df_all, building_annual_csv, var=var)

Total estimated kWh: 2382364.957421679
Annual scaling detail saved to ../data/outputs/annual_monthly/meter_annual_scaling_detail_fy26_AURORA.csv
Annual kwh saved to ../data/outputs/annual_monthly/meter_annual_kwh_fy26_AURORA.csv
Building annual kwh file saved to ../data/outputs/annual_monthly/building_annual_kwh_fy26_AURORA.csv


### 7. Calculate Monthly kWh

Same end-minus-start calculation as Section 5, just run once per calendar month instead of once for the whole year
- Special meters are scaled using a per month threshold (**monthly_min_valid_fraction**), not the fixed 5-month one used for the annual number
- This is why a meter's monthly totals can add up to something slightly different than its annual total

Output:
- `meter_monthly_scaling_detail_fyXX.csv`: long format, one row per meter/month (raw_difference, difference, R², info, % scaled, estimated_kwh)
- `meter_monthly_pct_scaled_fyXX.csv`: wide format, one row per meter, one column per month, values = % scaled
- `meter_monthly_long_fyXX.csv`: long format for Tableau (meter_name, month, kwh, pct_estimated)
- `meter_monthly_kwh_fyXX.csv`: wide format, one row per meter, one column per month, values = kwh


##### <span style="color:royalblue">Key for % scaled values:</span>
- **(blank)**: No scaling was reported; no estimation was needed (exactly 0%). The kWh value for that meter/month is a real, unadjusted number.
- **a percentage (e.g. 0.4%, 12.3%)**: That share of the month was estimated rather than measured, shown to one decimal place. The kWh value is still usable, just partly estimated.
- **N/A**: The meter had a known data issue that month (missing reading, meter restart, stuck value, etc.) and there wasn't enough valid data to estimate a value at all. The kWh value for that meter/month is also **blank** in `meter_monthly_kwh_fyXX.csv`, it could not be calculated.

##### <span style="color:royalblue">Key for % scaled values:</span>
| % Scaled value | Meaning |
|---|---|
| **(blank)** | No scaling was reported; no estimation was needed (exactly 0%). The kWh value for that year or month is a real, unadjusted number. |
| **a percentage (e.g. 0.4%, 12.3%)** | That share of the year or month was estimated rather than measured, shown to one decimal place. The kWh value is still usable, just partly estimated. |
| **N/A** | The meter had a known data issue that year or month (missing reading, meter restart, stuck value, etc.) and there wasn't enough valid data to estimate a value at all. The kWh value for that meter/month is also **blank** in `meter_monthly_kwh_fyXX.csv`, it could not be calculated. |

In [ ]:
### Compute kWh difference between End point and Start point for each calendar month ###
### Scaling applied per-month for missing-edge (R²=-1) meters using monthly_min_valid_fraction ###

# Step 1: Compute monthly differences (long format: one row per meter per month)
monthly_result_df = dc.compute_monthly_meter_differences(
    data_corrected,
    start_time,
    end_time,
    df_special_meters,
    df_restarts,
    monthly_min_valid_fraction=monthly_min_valid_fraction,
    r2_threshold=r2_threshold,
    restarts_thres=restarts_thres,
)

################ MONTHLY SCALING DETAIL ################
# Long format: one row per meter per month (matches meter_annual_scaling_detail_fyXX.csv's columns)
monthly_scaling_detail = monthly_result_df[[
    "meter_name", "month", "raw_difference", "difference", "R²", "info", "% scaled"
]].copy()

monthly_scaling_detail["estimated_kwh"] = (
    monthly_scaling_detail["difference"] - monthly_scaling_detail["raw_difference"]
)

monthly_scaling_detail = monthly_scaling_detail.sort_values(["meter_name", "month"]).reset_index(drop=True)

monthly_scaling_detail.to_csv(scaling_monthly_detail_csv, index=False)
print("Total estimated kWh (monthly):", monthly_scaling_detail["estimated_kwh"].sum())
print(f"Monthly scaling detail file saved to {scaling_monthly_detail_csv}")


# Wide format: one row per meter, one column per month, values = "% scaled"
monthly_pct_scaled = dc.export_monthly_scaling_detail(
    monthly_result_df, monthly_pct_scaled_csv
)

# Long format: one row per meter per month, for Tableau, values = "% scaled"
monthly_long = monthly_result_df[["meter_name", "month", "difference", "% scaled"]].copy()
monthly_long = monthly_long.rename(columns={"difference": "kwh", "% scaled": "pct_estimated"})
monthly_long["kwh"] = monthly_long["kwh"].round(1)
monthly_long = monthly_long.sort_values(["meter_name", "month"]).reset_index(drop=True)

monthly_long.to_csv(monthly_long_csv, index=False)
print(f"Monthly % scaled long format file saved to {monthly_long_csv}")

monthly_long.head()

##########################################################


# Step 2: Export kwh meter-level monthly differences -> one row per meter, one column per month (CSV)
monthly_long_df = dc.export_monthly_meter_differences(
    monthly_result_df, meter_info_file, meter_monthly_csv, var=var
)


Total estimated kWh (monthly): 3794054.513191943
Monthly scaling detail file saved to ../data/outputs/annual_monthly/meter_monthly_scaling_detail_fy26_AURORA.csv
Monthly % scaled wide format file saved to ../data/outputs/annual_monthly/meter_monthly_pct_scaled_fy26_AURORA.csv
Monthly % scaled long format file saved to ../data/outputs/annual_monthly/meter_monthly_long_fy26_AURORA.csv
Monthly kwh file saved to ../data/outputs/annual_monthly/meter_monthly_kwh_fy26_AURORA.csv


##### <span style="color:royalblue">Annual vs Monthly kwh:</span>

In [ ]:
### Compare each meter's annual difference to the sum of its monthly differences ###

annual_vs_monthly_check = dc.export_annual_vs_monthly_check(
    df_all, monthly_long_df, annual_monthly_check_csv, var=var
)

annual_vs_monthly_check.reindex(
    annual_vs_monthly_check[f"difference_{var}"].abs().sort_values(ascending=False).index
).head(10)

Annual vs monthly file saved to ../data/outputs/annual_monthly/meter_check_annual_vs_monthly_fy26_AURORA.csv


,meter_name,annual_kwh,sum_monthly_kwh,difference_kwh
12,biomedical_science_main_a,5694428.0,4033456.0,1660972.0
128,st_john_plant_science_main,2388052.0,1705436.0,682616.0
63,hamilton_lib_ph_iii_main_1,2064823.0,1431486.1,633336.8
10,biomedical_science_ch_1,1362127.0,815284.0,546843.0
14,biomedical_science_mcc_a,1470156.0,994000.0,476156.0
91,les_murakami_stadium_main,639245.0,307818.0,331427.0
19,bus_ad_shidler_main,1173132.0,843356.0,329776.0
76,hig_substation_3_main,721907.3,392355.0,329552.3
99,marine_science_mcc,629791.0,366999.8,262791.1
6,ag_science_mcc,863133.0,608037.0,255096.0


### 8. Insert Annual Building kWh into Master Sheet

##### <span style="color:royalblue">Insert data into master_sheet:</span>
- Fills the `annual_kwh` column of `<fy>_annual_kwh.csv` by matching `meter_name` against this run's computed annual differences, then saves it back in place
  - fills even if meter is a sub meter
- A blank `annual_kwh` cell can mean two different things, so they're reported separately:
  - the `meter_name` didn't match anything in this run's computed data (typo, retired/added meter), or
  - the `meter_name` matched fine, but that meter's computed annual kWh is itself `NaN` (e.g. a special meter without enough valid data this run)
- Computed `meter_name`s missing from the extract entirely are also reported

In [ ]:
### Fill this fiscal year's annual_kwh template with computed kWh difference (annual usage) ###

if Insert:
    # annual_kwh_extract_file = input_dir + FY.lstrip("_") + "_annual_" + var + ".csv"

    # TODO for aurora: (rem if doing harvest)
    annual_kwh_extract_file = input_dir + "fy26" + "_annual_" + var + ".csv"


    extract_template = pd.read_csv(annual_kwh_extract_file)
    col = f"annual_{var}"

    computed_values = df_all.set_index("meter_name")["difference"].round(1)
    extract_template[col] = extract_template["meter_name"].map(computed_values)

    extract_template.to_csv(annual_kwh_extract_file, index=False)

    template_meters = set(extract_template["meter_name"])
    computed_meters = set(df_all["meter_name"])

    unmatched_in_template = sorted(template_meters - computed_meters)
    unmatched_in_computed = sorted(computed_meters - template_meters)

    # Meters whose name DID match but whose computed annual kWh is itself NaN
    # (e.g. a special meter with too little valid data this run) distinct from
    # a name mismatch, so callers don't mistake "no data" for "not matched".
    matched_no_value = sorted(
        extract_template.loc[
            extract_template["meter_name"].isin(computed_meters) & extract_template[col].isna(),
            "meter_name",
        ]
    )

    # meter listed in template but not in computed meter data
    if unmatched_in_template:
        print(f"{len(unmatched_in_template)} meter_name(s) in {annual_kwh_extract_file} had no match in this run (left blank):")
        for name in unmatched_in_template:
            print(f"  - {name}")

    # meter in both template and computed data, but the computed value is NaN 
    if matched_no_value:
        print(f"{len(matched_no_value)} meter_name(s) matched but had no computable annual kWh this run (left blank):")
        for name in matched_no_value:
            print(f"  - {name}")

    # meter in computed data but not in template
    if unmatched_in_computed:
        print(f"{len(unmatched_in_computed)} computed meter_name(s) are not present in {annual_kwh_extract_file} (not written anywhere):")
        for name in unmatched_in_computed:
            print(f"  - {name}")

    extract_template.head()

13 meter_name(s) matched but had no computable annual kWh this run (left blank):
  - ching_complex_main
  - gilmore_hall_main_b
  - hper_klum_gym
  - it_center_main
  - korean_studies_main
  - malama_1_2_ehso_main
  - parking_struct_ph_i_main
  - pbrc_main_b
  - pope_lab_main
  - quad_chiller_plant_main
  - saunders_hall_main_a
  - sinclair_lib_main
  - softball_tennis_main
30 computed meter_name(s) are not present in ../data/extracts/fy26_annual_kwh.csv (not written anywhere):
  - ag_engineering_mcc
  - ag_science_mcc
  - biomedical_science_ch_1
  - biomedical_science_ch_2
  - biomedical_science_mcc_a
  - building_1171f_cds
  - everly_hall_main
  - gilmore_hall_mcc
  - hale_noelani_tower_a_b
  - hale_noelani_tower_b
  - hale_noelani_tower_c
  - hale_noelani_tower_c_d
  - hale_noelani_tower_e
  - hale_wainani_f_tower_main
  - hale_wainani_g_tower_main
  - hale_wainani_h_tower_main
  - hale_wainani_i_tower_main
  - hamilton_lib_ph_iii_ch_1
  - hamilton_lib_ph_iii_ch_2
  - hamilton_lib_p